In [ ]:
import torch
import gzip
import logging
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from ts2.data.transforms import HistologyTransform
from ts2.data.db_improc import instantiate_process_read
from ts2.data.meta_parser import CachedCSVParser
import einops
import io
import base64
from PIL import Image
from os.path import join as opj
import json

In [ ]:
from sklearn.manifold import TSNE
import altair as alt
import pandas as pd

In [ ]:
def get_knn_logits(cf, train_predictions, val_predictions):
    if "logits" in val_predictions:
        logging.warning("found existing logits. deleting for re-knn eval")
        del val_predictions["logits"]
    device = "cpu" #torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    #torch.cuda.empty_cache()
    with torch.no_grad():

        train_embs = torch.nn.functional.normalize(
            train_predictions["embeddings"], p=2, dim=1).T.to(device)
        val_embs = torch.nn.functional.normalize(val_predictions["embeddings"],
                                                 p=2,
                                                 dim=1)
        train_labels = train_predictions["label"].to(device)
        batch_size = cf["testing"]["knn"]["knn_params"]["batch_size"]
        all_scores = []

        for k in tqdm(range(val_embs.shape[0] // batch_size + 1)):
            # find current minibatch
            start_coeff = batch_size * k
            end_coeff = min(batch_size * (k + 1),
                            val_embs.shape[0])  # leftover
            val_embs_k = val_embs[start_coeff:end_coeff].to(
                device)  # 1536 x 2048

            # knn predict on the minibatch
            _, pred_scores = knn_predict(
                val_embs_k,
                train_embs,
                train_labels,
                len(set(train_predictions["label"].tolist())),
                knn_k=cf["testing"]["knn"]["knn_params"]["k"],
                knn_t=cf["testing"]["knn"]["knn_params"]["t"])

            # add to list
            all_scores.append(
                torch.nn.functional.normalize(pred_scores, p=1, dim=1).cpu())
            #torch.cuda.empty_cache()

    # add to predictions dict
    val_predictions["logits"] = torch.vstack(all_scores)
    return val_predictions


# code for kNN prediction from here:
# https://colab.research.google.com/github/facebookresearch/moco/blob/colab-notebook/colab/moco_cifar10_demo.ipynb
def knn_predict(feature, feature_bank, feature_labels, classes: int,
                knn_k: int, knn_t: float):
    """Helper method to run kNN predictions on features based on a feature bank
    Args:
        feature: Tensor of shape [B, D] consisting of N D-dimensional features
        feature_bank: Tensor of shape [N, D], a database of features used for kNN
        feature_labels: Labels for the features in our feature_bank
        classes: Number of classes (e.g. 10 for CIFAR-10)
        knn_k: Number of k neighbors used for kNN
        knn_t: Temperature in kNN, low temperature leads to more weighted kNN.
    """
    # compute cos similarity between each feature vector and feature bank ---> [B, N]
    sim_matrix = torch.mm(feature, feature_bank)
    # [B, K]
    sim_weight, sim_indices = sim_matrix.topk(k=knn_k, dim=-1)
    # [B, K]
    sim_labels = torch.gather(feature_labels.expand(feature.shape[0], -1),
                              dim=-1,
                              index=sim_indices)
    # we do a reweighting of the similarities
    sim_weight = (sim_weight / knn_t).exp()
    # counts for each class
    one_hot_label = torch.zeros(feature.shape[0] * knn_k,
                                classes,
                                device=sim_labels.device)
    # [B*K, C]
    one_hot_label = one_hot_label.scatter(dim=-1,
                                          index=sim_labels.view(-1, 1),
                                          value=1.0)
    # weighted score ---> [B, C]
    pred_scores = torch.sum(one_hot_label.view(feature.size(0), -1, classes) *
                            sim_weight.unsqueeze(dim=-1),
                            dim=1)
    pred_labels = pred_scores.argsort(dim=-1, descending=True)
    return pred_labels, pred_scores

In [ ]:
def im_to_bytestr(image) -> str:
        output = io.BytesIO()
        Image.fromarray(image).save(output, format='JPEG')
        return "data:image/jpeg;base64," + base64.b64encode(
            output.getvalue()).decode()

class SRHCellImageGetter():

    def __init__(self, data_root="/nfs/turbo/umms-tocho-snr/exp/chengjia/scsrh_repl_root2/", cached_parser_files=None, cf=None):
        super().__init__()
        
        if not cached_parser_files:
            cached_parser_files = [
                "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/data/tsmeta/scsrh7_patchcode_label/5a6d4937_cell40_nump_train",
                "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/data/tsmeta/scsrh7_patchcode_label/5a6d4937_cell40_nump_trainval"
            ]
        if cf:
            curr_base_aug_params =  cf.data.transform.test.params.base_aug_params
            curr_base_aug_params["to_uint8"] = True
            curr_base_aug_params["mask_params"]["how_to_process"]="small_patch"
        else:

            curr_base_aug_params = {
                "laser_noise_config": None,
                "get_third_channel_params": {
                    "mode": "three_channels",
                    "subtracted_base":0.07629394531
                },
                "mask_params":{
                    "how_to_process": "small_patch"
                },
                "to_uint8": True
            }
        self.transform_ = HistologyTransform(
            which_set="scsrh",
            base_aug_params=curr_base_aug_params,
            strong_aug_params={
                "aug_list": [
                    {
                        "which": "center_crop_always_apply",
                        "params":{
                          "size": 48
                        }
                    }
                ],
                "aug_prob": 1
            })

        self.process_read_ = instantiate_process_read(
            which="cell_memmap",
            which_set="scsrh")

        self.all_tsm_ = {}
        for k in cached_parser_files:
            self.all_tsm_.update(
                CachedCSVParser(k)()[-1])
        self.data_root_ = data_root

    def make_im_path(self, x):
        return opj(self.data_root_, x)

    def get_im(self, im_row):
        mmap_info = self.all_tsm_[im_row["slide"]]
        mmap_path = self.make_im_path(mmap_info["path"])
        im = self.process_read_(mmap_path, tuple(mmap_info["shape"]),
                                int(im_row["path"].split("@")[-1]))
        return einops.rearrange(self.transform_(im), "c h w -> h w c").numpy()

class SCBenchImageGetter():

    def __init__(self, data_root="/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/scbench/",
                 slides_file="/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/data/data/scbench/mosaic.csv",
                 cf=None):
        super().__init__()
        if cf:
            curr_base_aug_params =  cf.data.transform.test.params.base_aug_params
            curr_base_aug_params["to_uint8"] = True
            curr_base_aug_params["mask_params"]["how_to_process"]="small_patch"
        else:

            curr_base_aug_params = {
                "laser_noise_config": None,
                "get_third_channel_params": {
                    "mode": "three_channels",
                    "subtracted_base":0.07629394531
                },
                "mask_params":{
                    "how_to_process": "small_patch"
                },
                "to_uint8": True
            }
        

        
        self.transform_ = HistologyTransform(
            which_set="scsrh",
            base_aug_params=curr_base_aug_params,
            strong_aug_params={
                "aug_list": [
                    {
                        "which": "center_crop_always_apply",
                        "params":{
                          "size": 48
                        }
                    }
                ],
                "aug_prob": 1
            })


        all_meta = []
        all_images = []
        all_paths = []

        for _, s in pd.read_csv(slides_file).iterrows():
            meta = pd.read_csv(f"{data_root}/scbench_processed/{s['mosaic']}.csv")
            all_meta.append(meta["annot_labels"])
            all_images.append(
                torch.load(f"{data_root}/scbench_processed/{s['mosaic']}.pt").to(torch.float))
            all_paths.extend([f"scbench.{s['ttype']}.{s['mosaic']}@{i}" for i in range(len(meta))])

        all_meta = pd.concat(all_meta)
        all_images = torch.cat(all_images)

        self.instances_ = {k:j
                           for j,k in zip(all_images, all_paths)}


    def get_im(self, im_row):

        return einops.rearrange(self.transform_(self.instances_[im_row["path"]]),
                                 "c h w -> h w c").numpy()



In [ ]:
def get_predictions(train, trainval):
    with gzip.open(opj(train, "predictions/train_predictions.pt.gz")) as fd:
        db = torch.load(fd)
    with gzip.open(opj(trainval, "predictions/val_predictions.pt.gz")) as fd:
        test = torch.load(fd)
        
    return db, test

In [ ]:
def compute_emb_sim_top_k(query_feat: torch.Tensor,
                       tgt_feat: torch.Tensor,
                       k: int,
                       mask: torch.Tensor = None, device="cpu"):

    query_feat, tgt_feat = query_feat.to(device), tgt_feat.to(device)
    if mask is not None: mask = mask.to(device)

    sim = (torch.nn.functional.normalize(query_feat, p=2, dim=1)
        @ torch.nn.functional.normalize(tgt_feat, p=2, dim=1).T)
    if mask is not None: sim[mask.T] = 0

    logging.info(f"Sim matrix shape {sim.shape}")
    return sim.argsort(dim=1, descending=True)[:, :k]

In [ ]:
# dinov2+distil
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/dc0bfcd4_May02-23-35-53_sd1000_dev_tune0/models/eval/training_119999/75979ced_May14-12-03-12_sd1000_smpt24_SEBENCHv2_epoch0-step124999/predictions/train_predictions.pt.gz") as fd:
#    db = torch.load(fd)

#dinov2
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/2e7833f0_Apr30-16-49-29_sd1000_dev_tune0/models/eval/training_124999/86c3362e_May14-16-51-32_sd1000_smpt_SEBENCHv2_epoch0-step124999/predictions/train_predictions.pt.gz") as fd:
#    db = torch.load(fd)

# dinov2+distil?
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/dc0bfcd4_May02-23-35-53_sd1000_dev_tune0/models/eval/training_119999/14425119_May14-00-09-43_sd1000_smpt_SEBENCHv2_epoch0-step124999/predictions/train_predictions.pt.gz") as fd:
#    db = torch.load(fd)


# dinov2 with instance norm
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/2e7833f0_Apr30-16-49-29_sd1000_dev_tune0/models/eval/training_124999/2088bbca_May15-17-18-14_sd1000_smpt_SEBENCHv2_epoch0-step124999/predictions/train_predictions.pt.gz") as fd:
#    db = torch.load(fd)


# dinov2+distil with instance norm
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/9baaa202_May06-17-06-06_sd1000_dev_tune0/models/eval/training_124999/ca1a1742_May15-17-33-16_sd1000_smpt_SEBENCHv2_epoch0-step124999/predictions/train_predictions.pt.gz") as fd:
#    db = torch.load(fd)

In [ ]:
# dinov2+distil
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/dc0bfcd4_May02-23-35-53_sd1000_dev_tune0/models/eval/training_119999/447b93c9_May14-13-27-43_sd1000_smpt24_SEBENCHv2_epoch0-step124999/predictions/val_predictions.pt.gz") as fd:
#    test = torch.load(fd)

#dinov2
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/2e7833f0_Apr30-16-49-29_sd1000_dev_tune0/models/eval/training_124999/f399da5e_May14-16-47-03_sd1000_smpt_SEBENCHv2_epoch0-step124999/predictions/val_predictions.pt.gz") as fd:
#    test = torch.load(fd)

# dinov2+distil?
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/dc0bfcd4_May02-23-35-53_sd1000_dev_tune0/models/eval/training_119999/831b9e34_May13-23-42-43_sd1000_smpt_SEBENCHv2_epoch0-step124999/predictions/val_predictions.pt.gz") as fd:
#    test = torch.load(fd)

# dinov2 with instance norm
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/2e7833f0_Apr30-16-49-29_sd1000_dev_tune0/models/eval/training_124999/613d331c_May15-17-20-30_sd1000_smpt_SEBENCHv2_epoch0-step124999/predictions/val_predictions.pt.gz") as fd:
#    test = torch.load(fd)
    

# dinov2+distil with instance norm
#with gzip.open("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_hdinov2_tile/9baaa202_May06-17-06-06_sd1000_dev_tune0/models/eval/training_124999/6acf826d_May15-17-45-47_sd1000_smpt_SEBENCHv2_epoch0-step124999/predictions/val_predictions.pt.gz") as fd:
#    test = torch.load(fd)

In [ ]:
#db, test = get_predictions(
#    train=   "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/58feb8b9_May20-21-18-52_sd1000_dev_tune0/models/eval/training_59999/57a53d66_May21-15-58-26_sd1000_smpt_SEBENCHv2_epoch0-step59999_tune0",
#    trainval="/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/58feb8b9_May20-21-18-52_sd1000_dev_tune0/models/eval/training_59999/c368f053_May21-16-27-07_sd1000_smpt_SEBENCHv2_epoch0-step59999_tune0"
#)

#db, test = get_predictions(
#    train=   "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/e89a6de5_May19-22-29-25_sd1000_dev_tune4/models/eval/training_79999/431bb568_May23-15-32-24_sd1000_smpt_SRH7_SEPATCH_epoch0-step79999_tune0",
#    trainval="/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/e89a6de5_May19-22-29-25_sd1000_dev_tune4/models/eval/training_79999/431bb568_May23-15-32-24_sd1000_smpt_SRH7_SEPATCH_epoch0-step79999_tune0"
#)

db, test = get_predictions(
    train=   "/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/72a4fc4d_May23-09-34-36_sd1000_dev_tune0/models/eval/training_124999/2177c700_May26-11-11-36_sd1000_smpt_BENCHDB_SE_epoch0-step124999_tune2",
    trainval="/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/72a4fc4d_May23-09-34-36_sd1000_dev_tune0/models/eval/training_124999/55448b0f_May26-03-48-24_sd1000_smpt_BENCH_SE_epoch0-step124999_tune2"
)




In [ ]:
db_emb = torch.nn.functional.normalize(db["embeddings"])
test_emb = torch.nn.functional.normalize(test["embeddings"])

In [ ]:
knn_conf = {
    "testing":{"knn":{"knn_params":{
        "batch_size":1024,
        "k": 20,
        "t": 0.07
}}}}

val_predictions = get_knn_logits(knn_conf, db, test)

In [ ]:
pred_label = val_predictions["logits"].argmax(axis=1)
gt_label = val_predictions["label"]

n_out_class = len(set(db["label"].tolist()))
n_in_class = len(set(test["label"].tolist()))

cm = np.zeros((n_in_class, n_out_class))
for i,j in zip(gt_label, pred_label):
    cm[i,j]+=1

In [ ]:
cm

In [ ]:
cm/np.expand_dims(cm.sum(axis=1),axis=1)

In [ ]:
db_class_names = ["normal", "tumor"]
test_class_names = ['Astrocytes', 'Neurons', 'Oligodendrocytes', 'RAstrocytes', 'Giant Cell', 'Glioma', 'Lymphocytes', 'Macrophages', 'Adeno', 'Melanoma', 'Sarcoma', 'Squamous', 'Mitotic']

db_class_names = ["HGG", "LGG", "Mening", "Mets", "Normal", "Pit", "Schwan"]
#test_class_names = db_class_names


In [ ]:
fig, ax = plt.subplots(1,1)
ax.imshow(cm/np.expand_dims(cm.sum(axis=1),axis=1))
ax.set_xticklabels(db_class_names)
ax.set_yticklabels(test_class_names)
ax.set_yticks(np.arange(len(test_class_names)))
ax.set_xticks(np.arange(len(db_class_names)))

In [ ]:
sbig = SCBenchImageGetter()
scig = SRHCellImageGetter()


In [ ]:
db["label"]

In [ ]:
if len(db["embeddings"]) > 10000:
    db_idx = sorted(np.random.permutation(db["embeddings"].shape[0])[:10000])
    db_embs = db["embeddings"][db_idx]
    db_labels = np.array([db_class_names[i] for i in db["label"]])[db_idx]
    db_path = np.array(db["path"])[db_idx]
else:
    db_embs = db["embeddings"]
    db_labels = np.array([db_class_names[i] for i in db["label"]])
    db_path = np.array(db["path"])

In [ ]:
tsne = TSNE(n_components=2)
xy = tsne.fit_transform(torch.cat((db_embs, test["embeddings"])))

In [ ]:
all_labels = np.concatenate((db_labels, np.array([test_class_names[i] for i in test["label"]])))
all_paths = np.concatenate((db_path, np.array(test["path"])))
#all_ims = [im_to_bytestr(scig.get_im({"path":i, "slide": "-".join(i.split("-")[:2])})) for i in tqdm(db_path)
#          ] + [im_to_bytestr(sbig.get_im({"path": i})) for i in tqdm(test["path"])]
all_ims = [im_to_bytestr(scig.get_im({"path":i, "slide": "-".join(i.split("-")[:2])})) for i in tqdm(db_path)
          ] + [im_to_bytestr(sbig.get_im({"path":i, "slide": "-".join(i.split("-")[:2])})) for i in tqdm(np.array(test["path"]))]
data = pd.DataFrame({"x": xy[:,0], "y": xy[:,1], "label":all_labels, "path": all_paths, "image": all_ims})

In [ ]:
alt.data_transformers.disable_max_rows()
class_selection = alt.selection_point(fields=["label"], bind='legend')
chart = alt.Chart(
    data
).mark_circle(
    filled=True
).encode(
    x="x",
    y="y",
    color="label",
    tooltip=["image", "path", "label"],
    opacity=alt.condition(
            class_selection,
            alt.value(1.0), alt.value(0.01)
    )
).add_params(
    class_selection
).properties(
    width=700,
    height=700
)

chart.interactive()

In [ ]:
chart.interactive().save("tsne_58feb8b9.html")

In [ ]:
dbdf = pd.DataFrame({
     "embeddings":db_emb.tolist(),
     "labels":  db["label"].tolist(),
     "path":    db["path"]
 })
testdf = pd.DataFrame({
     "embeddings":test_emb.tolist(),
     "labels":  test["label"].tolist(),
     "path":    test["path"]
 })

In [ ]:
testdf

In [ ]:
def safe_sample(x, n=8):
    return x.sample(n=n if len(x) >= n else len(x))


nn_viz_cf={
    "exclude_same_patient": True,
    "k": 63
}

dbdf["slide"] = dbdf["path"].str.extract("((?:INV|NIO)_[a-zA-Z]+_[0-9]+[a-zA-Z]?-[0-9a-zA-Z]*)")
dbdf["pt"] = dbdf["path"].str.extract("((?:INV|NIO)_[a-zA-Z]+_[0-9]+[a-zA-Z]?)")
testdf["slide"] = testdf["path"].str.extract("((?:INV|NIO)_[a-zA-Z]+_[0-9]+[a-zA-Z]?-[0-9a-zA-Z]*)")
testdf["pt"] = testdf["path"].str.extract("((?:INV|NIO)_[a-zA-Z]+_[0-9]+[a-zA-Z]?)")

query_pred = testdf.groupby(['labels'], group_keys=False).apply(safe_sample)
tgt_pred = dbdf

query_feat = torch.tensor([d for d in query_pred["embeddings"]])
tgt_feat = torch.tensor([d for d in tgt_pred["embeddings"]])

if nn_viz_cf["exclude_same_patient"]:
    query_pt = np.expand_dims(
        query_pred["pt"],0)
    tgt_pt = np.expand_dims(
        tgt_pred["pt"],1)
    mask = torch.tensor(tgt_pt == query_pt)
else:
    mask = None

#import pdb; pdb.set_trace()
sim = compute_emb_sim_top_k(query_feat,
                                 tgt_feat,
                                 k=nn_viz_cf["k"],
                                 mask=mask)


In [ ]:
chosen_dbdf = dbdf.iloc[sorted(set(sim.flatten().tolist()))]
chosen_dbdf["image"] = chosen_dbdf.apply(scig.get_im, axis=1).apply(im_to_bytestr)

In [ ]:
nn_im = [[chosen_dbdf.loc[i.item()]["image"] for i in j] for j in sim]
nn_p = [[chosen_dbdf.loc[i.item()]["path"] for i in j] for j in sim]
nn_lbl = [[db_class_names[chosen_dbdf.loc[i.item()]["labels"].item()] for i in j] for j in sim]
query_pred["nnim"] = nn_im
query_pred["nnid"] = query_pred['nnim'].apply(lambda x: list(range(len(x))))
query_pred["nnp"] = nn_p
query_pred["nnlbl"] = nn_lbl
query_pred["labels"] = query_pred["labels"].apply(lambda x: test_class_names[x])
query_pred_with_nn = query_pred.reset_index(drop=True).explode(["nnim", "nnid", "nnp", "nnlbl"])
#query_pred = query_pred.drop(["embeddings"], axis=1)
query_pred_with_nn = query_pred_with_nn.reset_index()

In [ ]:
#query_pred["image"] = query_pred.apply(scig.get_im, axis=1).apply(im_to_bytestr)
query_pred["image"] = query_pred.apply(sbig.get_im, axis=1).apply(im_to_bytestr)
query_pred["xval"] = -1
query_pred = query_pred.reset_index(drop=True).reset_index()

In [ ]:
query_pred

In [ ]:
image_size_slider = alt.binding_range(min=8, max=512, step=4, name='Image size:')
image_size = alt.param(name='size', value=32, bind=image_size_slider)

chart_query_lbl = alt.Chart(query_pred).mark_point(
    size=image_size**2,strokeWidth=10,shape="square"
).encode(
    x=alt.X("xval:Q"),
    y=alt.Y('index:Q', scale=alt.Scale(reverse=True)),
    color="labels:N",
).add_params(image_size)

chart_query = alt.Chart(query_pred).mark_image(width=image_size,
    height=image_size  
).encode(
    x=alt.X("xval:Q"),
    y=alt.Y('index:Q', scale=alt.Scale(reverse=True)),
    url='image:N',
    tooltip=['path', "xval", "index", "labels"],

).add_params(image_size)

chart_tgt_lbl = alt.Chart(query_pred_with_nn).mark_point(
    size=image_size**2,strokeWidth=10,shape="square"
).encode(
    x=alt.X('nnid:Q'),
    y=alt.Y('index:Q', scale=alt.Scale(reverse=True)),
    color="nnlbl:N",
).add_params(image_size)

chart_tgt = alt.Chart(query_pred_with_nn).mark_image(
    width=image_size,
    height=image_size  
).encode(
    x=alt.X('nnid:Q'),
    y=alt.Y('index:Q', scale=alt.Scale(reverse=True)),
    url='nnim:N',
    tooltip=['nnp'],
).add_params(image_size)

(chart_query_lbl+chart_query+chart_tgt_lbl+chart_tgt).properties(
    height=1800,
    width=1800
).interactive()

In [ ]:
(chart_query_lbl+chart_query+chart_tgt_lbl+chart_tgt).properties(
    height=1800
).interactive().save("nn2.html")

In [ ]:

def padding_func(x):
    return np.pad(np.array(x),
                  pad_width=((2, 2), (2, 2), (0, 0)),
                  constant_values=255)

im_xms = np.vstack([
    padding_func(self.get_im_impl_(x))
    for _, x in query_pred.iterrows()
])
im_divider = np.ones_like(im_xms) * 255
im_nns = np.vstack([
    np.hstack([
        padding_func(self.get_im_impl_(tgt_pred.iloc[int(j)]))
        for j in i
    ]) for i in tqdm(sim, desc="load NN image")
])
out = np.hstack([im_xms, im_divider, im_nns])
#imageio.imwrite(opj(self.out_root_, "nn.png"), out.astype(np.uint8))

In [ ]:
dbdf["path"].str.extract("((?:INV|NIO)_[a-zA-Z]+_[0-9]+[a-zA-Z]?-[0-9a-zA-Z]*)")

In [ ]:
chart.interactive().save("newval.html")

In [ ]:
def viz_seg_model(nion, slide):
    with open(f"/nfs/turbo/umms-tocho/root_srh_db/UM/{nion}/{nion}_meta.json"
              ) as fd:
        pt_meta = json.load(fd)

    patch_pred = pt_meta["slides"][slide]["predictions"]["03207B00"]
    patch_seg_preds = {
        item.removesuffix(".tif").removesuffix(".tiff").removeprefix(f"{nion}-{slide}-"):
        key
        for key, values in patch_pred.items()
        for item in values
    }
    patch_seg_preds = {
        tuple([int(i) for i in k.split("_")]): v
        for k, v in patch_seg_preds.items()
    }
    patch_seg_preds = {
        tuple([(k[0] + k[2]) // 100, (k[1] + k[3]) // 100]): v
        for k, v in patch_seg_preds.items()
    }
    viz_dim = (
        (np.array(list(patch_seg_preds.keys())).max(axis=0) // 10) + 1) * 10
    viz = np.zeros((*viz_dim, 3), dtype=np.uint8)
    colors = {
        "tumor": [255, 0, 0],
        "normal": [0, 255, 0],
        "nondiagnostic": [0, 0, 255]
    }
    for k, v in patch_seg_preds.items():
        viz[k[0]:k[0] + 3, k[1]:k[1] + 3, :] = colors[v]
    return viz

In [ ]:
plt.imshow(viz_seg_model("NIO_UM_1493", "1"))